# Single-Objective Bayesian Optimization: PR-AUC Baseline

**Purpose**: Research baseline for comparison with multi-objective approach

- **Objective**: Maximize PR-AUC (minimize negative PR-AUC)
- **Method**: BoTorch qExpectedImprovement
- **Budget**: 40 evaluations (16 initial + 6 iterations × 4 batch)
- **Comparison**: Load existing multi-objective results (NO recomputation)
- **Time**: ~60 hours (vs ~84 hours saved by reusing multi-objective)

## Key Differences from Multi-Objective

| Aspect | Multi-Objective | Single-Objective |
|--------|----------------|------------------|
| Objectives | 4D: PR-AUC, AUROC, Brier, Degradation | 1D: PR-AUC only |
| Acquisition | qNEHVI | qExpectedImprovement |
| GP Model | ModelListGP (4 GPs) | SingleTaskGP (1 GP) |
| Budget | 56 evals (DONE) | 40 evals (NEW) |
| Result | Pareto front | Single best value |

## Setup Requirements

**Before running:**
1. Enable GPU accelerator in Kaggle (Settings → Accelerator → GPU P100)
2. Add input datasets:
   - `core-code2` (project code)
   - `vindr-dataset-dave` (VinDr-Mammo images)
   - `dataaa` (CSV + split info + multi-objective results)
   - `inbreast-dataset` (INbreast DICOMs + XLS)
3. Run Cell 1 to install BoTorch (takes ~2 minutes)

**Important**: Cell 1 installs BoTorch. Run it first before other cells.

## Phase 0: Setup

In [ ]:
# ========================================
# INSTALL PACKAGES
# ========================================

# Install BoTorch and dependencies
!pip install -q botorch gpytorch

# Verify installation
import torch
import botorch
import gpytorch

print("✓ Package installation complete")
print(f"  PyTorch: {torch.__version__}")
print(f"  BoTorch: {botorch.__version__}")
print(f"  GPyTorch: {gpytorch.__version__}")

In [ ]:
# ========================================
# IMPORTS
# ========================================

import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import json
from datetime import datetime

# Add project to path
if os.path.exists('/kaggle/input/core-code2'):
    project_root = "/kaggle/input/core-code2"
else:
    project_root = Path.cwd()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import project modules
from breast_cancer_detection.src.datasets import (
    VinDRMammoBinaryDataset,
    create_breast_level_splits
)
from breast_cancer_detection.src.preprocessing import MammographyPreprocessor
from breast_cancer_detection.src.evaluation_functions import (
    train_and_evaluate,
    create_evaluation_function
)

# Import single-objective BoTorch utilities
from breast_cancer_detection.src.botorch_so import (
    SingleObjectiveGPModel,
    suggest_candidates_ei,
    initial_sobol_sampling
)
from breast_cancer_detection.src.botorch_utils import (
    HyperparameterTransform,
    BoTorchCheckpoint
)

# Import INbreast dataset loader
from breast_cancer_detection.src import (
    INbreastDatasetFromXLS,
    create_inbreast_calibration_test_splits
)

print("✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# ========================================
# CONFIGURATION
# ========================================

# Data paths (Kaggle)
DATA_ROOT = r"/kaggle/input/vindr-dataset-dave/vindr_mammo_dataset_dave/images"
CSV_FILE = r"/kaggle/input/dataaa/stratified_selection.csv"
INBREAST_DICOM_DIR = r"/kaggle/input/inbreast-dataset/INbreast Release 1.0/AllDICOMs"
INBREAST_XLS_FILE = r"/kaggle/input/inbreast-dataset/INbreast Release 1.0/INbreast.xls"
INBREAST_SPLIT_JSON = r"/kaggle/input/dataaa/inbreast_split_info.json"

# Optimization parameters
N_INITIAL = 16        # Initial Sobol samples (2× dimensionality is standard)
N_ITERATIONS = 6      # BO iterations (fewer than multi-objective's 10)
BATCH_SIZE = 4        # Candidates per iteration
TOTAL_BUDGET = N_INITIAL + N_ITERATIONS * BATCH_SIZE  # 16 + 24 = 40

# Output directory
OUTPUT_DIR = Path("/kaggle/working/results/botorch_so_pr_auc")
RUN_ID = f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training configuration
TRAIN_BATCH_SIZE = 4
NUM_WORKERS = 2
PATIENCE = 3  # Early stopping patience
MAX_EPOCHS = 7
RANDOM_SEED = 42

print(f"Configuration:")
print(f"  Device: {DEVICE}")
print(f"  Budget: {TOTAL_BUDGET} evaluations ({N_INITIAL} initial + {N_ITERATIONS}×{BATCH_SIZE} BO)")
print(f"  Output: {OUTPUT_DIR / RUN_ID}")
print(f"  Training: max {MAX_EPOCHS} epochs, patience={PATIENCE}")
print(f"\nTime Estimate:")
print(f"  Per evaluation: 60-120 minutes")
print(f"  Total: {TOTAL_BUDGET * 90 / 60:.1f} hours (avg)")
print(f"  Savings vs recomputing multi-objective: 84 hours")

In [ ]:
# ========================================
# LOAD VINDR-MAMMO DATASET
# ========================================

print("="*80)
print("LOADING VINDR-MAMMO DATASET")
print("="*80)

# Create preprocessor
preprocessor = MammographyPreprocessor(
    target_size=(720, 480),
    aspect_ratio=1.5
)

# Load VinDr-Mammo
vindr_dataset = VinDRMammoBinaryDataset(
    images_root=DATA_ROOT,
    csv_file=CSV_FILE,
    preprocessor=preprocessor
)

print(f"\nTotal samples: {len(vindr_dataset)}")

# Class distribution
all_labels = [vindr_dataset[i][1].item() for i in range(len(vindr_dataset))]
n_benign = sum([1 for l in all_labels if l == 0])
n_malignant = sum([1 for l in all_labels if l == 1])

print(f"  Benign: {n_benign}")
print(f"  Malignant: {n_malignant}")
print(f"  Ratio: {n_benign/n_malignant:.2f}:1")

# Create 80/20 breast-level split
train_dataset, val_dataset = create_breast_level_splits(
    dataset=vindr_dataset,
    train_ratio=0.8,
    random_state=RANDOM_SEED,
    stratify=True
)

# Pos weight for BCE loss
pos_weight = n_benign / n_malignant
print(f"\nPos weight for BCE loss: {pos_weight:.3f}")
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

In [ ]:
# ========================================
# LOAD INBREAST DATASET
# ========================================

print("="*80)
print("LOADING INBREAST WITH SAME SPLIT AS OPTIMIZATION")
print("="*80)

# Load split info from multi-objective run
with open(INBREAST_SPLIT_JSON, 'r') as f:
    split_info = json.load(f)

print(f"✓ Loaded split info from optimization run")
print(f"  Random seed: {split_info['random_state']}")

# Load INbreast dataset
inbreast_dataset = INbreastDatasetFromXLS(
    dicom_dir=INBREAST_DICOM_DIR,
    xls_file=INBREAST_XLS_FILE,
    preprocessor=preprocessor
)

# Recreate SAME split using saved random seed
inbreast_calibration, inbreast_test, recreated_split_info = create_inbreast_calibration_test_splits(
    dataset=inbreast_dataset,
    calibration_ratio=0.2,
    random_state=split_info['random_state'],
    stratify=True
)

print(f"✓ Split recreated with same random seed")
print(f"  Calibration: {len(inbreast_calibration)} images")
print(f"  Test: {len(inbreast_test)} images (held out)")

In [ ]:
# ========================================
# INITIALIZE HYPERPARAMETER TRANSFORM
# ========================================

transform = HyperparameterTransform()
bounds_normalized = transform.normalized_bounds

print("Hyperparameter bounds (normalized [0,1]):")
print(bounds_normalized)
print(f"\nNumber of variables: {bounds_normalized.shape[1]}")

## Phase 1: Initial Sampling

In [ ]:
# ========================================
# INITIALIZE RESULTS STORAGE
# ========================================

output_dir = OUTPUT_DIR / RUN_ID
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "evaluations.csv"
checkpoint_dir = output_dir / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# CSV columns (track all 4 metrics even though only optimizing PR-AUC)
csv_columns = [
    'evaluation_id', 'iteration', 'phase',
    'learning_rate', 'weight_decay', 'dropout',
    'augmentation_strength', 'unfreeze_fraction',
    'pr_auc', 'auroc', 'brier', 'cross_dataset_degradation',
    'neg_pr_auc',  # Objective being optimized (negative for minimization)
    'best_pr_auc_so_far'  # For convergence tracking
]

if csv_path.exists():
    # Resume from existing
    results_df = pd.read_csv(csv_path)
    print(f"✓ Resuming from {len(results_df)} existing evaluations")

    # Rebuild tensors for BoTorch
    X_linear = []
    for _, row in results_df.iterrows():
        x_linear = [
            np.log10(row['learning_rate']),
            np.log10(row['weight_decay']),
            row['dropout'],
            row['augmentation_strength'],
            row['unfreeze_fraction']
        ]
        X_linear.append(x_linear)

    X_linear_tensor = torch.tensor(X_linear, dtype=torch.float64)
    X_train = transform.to_normalized(X_linear_tensor)

    Y_train = torch.tensor(
        results_df['neg_pr_auc'].values.reshape(-1, 1),
        dtype=torch.float64
    )

    start_iteration = results_df[results_df['phase'] == 'bo']['iteration'].max()
    start_iteration = -1 if pd.isna(start_iteration) else int(start_iteration)
else:
    # Fresh start
    results_df = pd.DataFrame(columns=csv_columns)
    X_train = None
    Y_train = None
    start_iteration = -1

print(f"Results CSV: {csv_path}")
print(f"Starting from iteration: {start_iteration + 1}")

In [ ]:
# ========================================
# INITIAL SOBOL SAMPLING
# ========================================

if X_train is None or len(X_train) < N_INITIAL:
    n_needed = N_INITIAL - (len(X_train) if X_train is not None else 0)

    print(f"Generating {n_needed} Sobol samples...")
    X_initial = initial_sobol_sampling(
        n_vars=5,
        n_samples=n_needed,
        bounds=bounds_normalized
    )

    print(f"Sobol samples generated: {X_initial.shape}")
else:
    print(f"Already have {len(X_train)} initial samples, skipping Sobol generation")
    X_initial = None

In [ ]:
# ========================================
# EVALUATE INITIAL SAMPLES
# ========================================

if X_initial is not None:
    eval_id_start = len(results_df)
    best_pr_auc_so_far = results_df['pr_auc'].max() if len(results_df) > 0 else 0.0

    for i, x in enumerate(tqdm(X_initial, desc="Initial Sampling")):
        eval_id = eval_id_start + i

        print(f"\nEvaluation {eval_id}: Initial sample {i+1}/{len(X_initial)}")

        # Convert to real hyperparameters
        hyperparams = transform.to_real_hyperparams(x.cpu().numpy())

        print(f"  LR: {hyperparams['learning_rate']:.6f}")
        print(f"  WD: {hyperparams['weight_decay']:.6f}")
        print(f"  Dropout: {hyperparams['dropout']:.3f}")

        # Evaluate (train and get all 4 metrics)
        metrics = train_and_evaluate(
            hyperparams=hyperparams,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            device=DEVICE,
            batch_size=TRAIN_BATCH_SIZE,
            num_workers=NUM_WORKERS,
            patience=PATIENCE,
            max_epochs=MAX_EPOCHS,
            pos_weight=pos_weight,
            random_seed=RANDOM_SEED,
            inbreast_calibration_dataset=inbreast_calibration
        )

        # Compute objective (negative PR-AUC for minimization)
        neg_pr_auc = -metrics['pr_auc']

        # Update best so far
        best_pr_auc_so_far = max(best_pr_auc_so_far, metrics['pr_auc'])

        # Store results
        row = {
            'evaluation_id': eval_id,
            'iteration': -1,
            'phase': 'initial',
            'learning_rate': hyperparams['learning_rate'],
            'weight_decay': hyperparams['weight_decay'],
            'dropout': hyperparams['dropout'],
            'augmentation_strength': hyperparams['augmentation_strength'],
            'unfreeze_fraction': hyperparams['unfreeze_fraction'],
            'pr_auc': metrics['pr_auc'],
            'auroc': metrics['auroc'],
            'brier': metrics['brier'],
            'cross_dataset_degradation': metrics['cross_dataset_degradation'],
            'neg_pr_auc': neg_pr_auc,
            'best_pr_auc_so_far': best_pr_auc_so_far
        }

        results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)

        # Save after each evaluation (checkpoint)
        results_df.to_csv(csv_path, index=False)

        # Update training data
        X_new = x.unsqueeze(0)
        Y_new = torch.tensor([[neg_pr_auc]], dtype=torch.float64)

        if X_train is None:
            X_train = X_new
            Y_train = Y_new
        else:
            X_train = torch.cat([X_train, X_new], dim=0)
            Y_train = torch.cat([Y_train, Y_new], dim=0)

        print(f"  PR-AUC: {metrics['pr_auc']:.4f}")
        print(f"  AUROC: {metrics['auroc']:.4f}")
        print(f"  Brier: {metrics['brier']:.4f}")
        print(f"  Best so far: {best_pr_auc_so_far:.4f}")

    print(f"\nInitial sampling complete: {len(X_train)} evaluations")
    print(f"Best PR-AUC so far: {best_pr_auc_so_far:.4f}")

    # Save checkpoint
    checkpoint_path = checkpoint_dir / "checkpoint_initial.pt"
    torch.save({
        'X_train': X_train,
        'Y_train': Y_train,
        'iteration': -1,
        'phase': 'initial_complete'
    }, checkpoint_path)
    print(f"Checkpoint saved: {checkpoint_path}")

## Phase 2: Bayesian Optimization Loop

In [ ]:
# ========================================
# INITIALIZE SINGLE-OBJECTIVE GP MODEL
# ========================================

gp_model = SingleObjectiveGPModel(
    n_vars=5,
    bounds=bounds_normalized
)

# Add training data
if X_train is not None and len(X_train) > 0:
    gp_model.update_data(X_train, Y_train)
    print(f"✓ GP model initialized with {len(X_train)} training points")
    print(f"  X_train shape: {X_train.shape}")
    print(f"  Y_train shape: {Y_train.shape}")
else:
    print("⚠ No training data yet - initial sampling needed")

In [ ]:
# ========================================
# BAYESIAN OPTIMIZATION LOOP
# ========================================

for iteration in range(start_iteration + 1, N_ITERATIONS):
    print(f"\n{'='*80}")
    print(f"BO ITERATION {iteration + 1}/{N_ITERATIONS}")
    print(f"{'='*80}")

    # Fit GP on current data
    print(f"Fitting GP on {len(gp_model.train_X)} points...")
    gp_model.fit_model()
    print(f"✓ GP fitted")

    # Suggest next batch using qEI
    print(f"Optimizing acquisition function (batch_size={BATCH_SIZE})...")
    X_next, acq_value = suggest_candidates_ei(
        gp_model=gp_model,
        batch_size=BATCH_SIZE,
        num_restarts=20,
        raw_samples=512
    )

    print(f"✓ Generated {len(X_next)} candidates")
    print(f"  Acquisition value: {acq_value.item():.6f}")

    # Evaluate candidates
    eval_id_start = len(results_df)
    best_pr_auc_so_far = results_df['pr_auc'].max()

    for i, x in enumerate(X_next):
        eval_id = eval_id_start + i

        print(f"\nEvaluating candidate {i+1}/{BATCH_SIZE} (ID: {eval_id})...")

        # Convert to real hyperparameters
        hyperparams = transform.to_real_hyperparams(x.cpu().numpy())

        print(f"  LR: {hyperparams['learning_rate']:.6f}")
        print(f"  WD: {hyperparams['weight_decay']:.6f}")

        # Evaluate (train and get all 4 metrics)
        metrics = train_and_evaluate(
            hyperparams=hyperparams,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            device=DEVICE,
            batch_size=TRAIN_BATCH_SIZE,
            num_workers=NUM_WORKERS,
            patience=PATIENCE,
            max_epochs=MAX_EPOCHS,
            pos_weight=pos_weight,
            random_seed=RANDOM_SEED,
            inbreast_calibration_dataset=inbreast_calibration
        )

        # Compute objective (negative PR-AUC for minimization)
        neg_pr_auc = -metrics['pr_auc']

        # Update best so far
        best_pr_auc_so_far = max(best_pr_auc_so_far, metrics['pr_auc'])

        # Store results
        row = {
            'evaluation_id': eval_id,
            'iteration': iteration,
            'phase': 'bo',
            'learning_rate': hyperparams['learning_rate'],
            'weight_decay': hyperparams['weight_decay'],
            'dropout': hyperparams['dropout'],
            'augmentation_strength': hyperparams['augmentation_strength'],
            'unfreeze_fraction': hyperparams['unfreeze_fraction'],
            'pr_auc': metrics['pr_auc'],
            'auroc': metrics['auroc'],
            'brier': metrics['brier'],
            'cross_dataset_degradation': metrics['cross_dataset_degradation'],
            'neg_pr_auc': neg_pr_auc,
            'best_pr_auc_so_far': best_pr_auc_so_far
        }

        results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)

        # Save after each evaluation (checkpoint)
        results_df.to_csv(csv_path, index=False)

        # Update GP training data
        gp_model.update_data(
            x.unsqueeze(0),
            torch.tensor([[neg_pr_auc]], dtype=torch.float64)
        )

        print(f"  PR-AUC: {metrics['pr_auc']:.4f}")
        print(f"  AUROC: {metrics['auroc']:.4f}")
        print(f"  Brier: {metrics['brier']:.4f}")
        print(f"  Cross-dataset deg: {metrics['cross_dataset_degradation']:.4f}")

    # Save checkpoint after iteration
    checkpoint_path = checkpoint_dir / f"checkpoint_iter_{iteration}.pt"
    BoTorchCheckpoint.save(
        path=checkpoint_path,
        gp_model=gp_model,
        X_train=gp_model.train_X,
        Y_train=gp_model.train_Y,
        iteration=iteration,
        best_pr_auc_so_far=best_pr_auc_so_far
    )

    print(f"\n✓ Iteration {iteration + 1} complete")
    print(f"  Best PR-AUC so far: {best_pr_auc_so_far:.4f}")
    print(f"  Checkpoint saved: {checkpoint_path}")

print("\n" + "="*80)
print("OPTIMIZATION COMPLETE")
print("="*80)

In [ ]:
# ========================================
# FINAL SUMMARY
# ========================================

# Load final results
results_df = pd.read_csv(csv_path)

# Find best solution
best_idx = results_df['pr_auc'].idxmax()
best_solution = results_df.loc[best_idx]

print("BEST SOLUTION FOUND:")
print("="*80)
print(f"Evaluation ID: {best_solution['evaluation_id']:.0f}")
print(f"Phase: {best_solution['phase']}")
if best_solution['phase'] == 'bo':
    print(f"BO Iteration: {best_solution['iteration']:.0f}")

print(f"\nHyperparameters:")
print(f"  Learning rate:        {best_solution['learning_rate']:.9f}")
print(f"  Weight decay:         {best_solution['weight_decay']:.9f}")
print(f"  Dropout:              {best_solution['dropout']:.3f}")
print(f"  Aug strength:         {best_solution['augmentation_strength']:.3f}")
print(f"  Unfreeze fraction:    {best_solution['unfreeze_fraction']:.3f}")

print(f"\nMetrics:")
print(f"  PR-AUC:              {best_solution['pr_auc']:.4f} ⭐")
print(f"  AUROC:               {best_solution['auroc']:.4f}")
print(f"  Brier:               {best_solution['brier']:.4f}")
print(f"  Cross-dataset deg:   {best_solution['cross_dataset_degradation']:.4f}")

## Phase 3: Comparison with Multi-Objective

In [ ]:
# ========================================
# LOAD EXISTING MULTI-OBJECTIVE RESULTS
# ========================================

print("="*80)
print("LOADING MULTI-OBJECTIVE RESULTS (Completed Experiment)")
print("="*80)

# Path to existing multi-objective results
MO_RESULTS_PATH = "/kaggle/input/dataaa/evaluations (10).csv"

if not os.path.exists(MO_RESULTS_PATH):
    raise FileNotFoundError(
        f"Multi-objective results not found: {MO_RESULTS_PATH}\n"
        f"Expected CSV from completed multi-objective run."
    )

mo_results = pd.read_csv(MO_RESULTS_PATH)
print(f"✓ Loaded multi-objective results: {len(mo_results)} evaluations")

# Extract Pareto front
def extract_pareto_efficient(df):
    """Extract Pareto-efficient solutions (4D minimization)."""
    # Detect column names (backward compatibility)
    degradation_col = 'cross_dataset_degradation'
    if 'robustness' in df.columns and 'cross_dataset_degradation' not in df.columns:
        degradation_col = 'robustness'
    elif 'robustness_degradation' in df.columns and 'cross_dataset_degradation' not in df.columns:
        degradation_col = 'robustness_degradation'

    # Convert to minimization (negate PR-AUC and AUROC)
    costs = df[['pr_auc', 'auroc', 'brier', degradation_col]].copy()
    costs['pr_auc'] = -costs['pr_auc']
    costs['auroc'] = -costs['auroc']

    # Pareto dominance
    is_efficient = np.ones(len(costs), dtype=bool)
    for i, c in enumerate(costs.values):
        if is_efficient[i]:
            is_efficient[is_efficient] = np.any(costs.values[is_efficient] < c, axis=1)
            is_efficient[i] = True

    return df[is_efficient].copy()

pareto_front = extract_pareto_efficient(mo_results)
print(f"✓ Extracted Pareto front: {len(pareto_front)} solutions")

# Best PR-AUC solution from multi-objective
mo_best_idx = pareto_front['pr_auc'].idxmax()
mo_best_solution = pareto_front.loc[mo_best_idx]

# Detect degradation column
if 'cross_dataset_degradation' in mo_best_solution:
    mo_degradation = mo_best_solution['cross_dataset_degradation']
elif 'robustness' in mo_best_solution:
    mo_degradation = mo_best_solution['robustness']
else:
    mo_degradation = mo_best_solution['robustness_degradation']

print(f"\nMulti-Objective Best PR-AUC Solution:")
print(f"  PR-AUC: {mo_best_solution['pr_auc']:.4f}")
print(f"  AUROC: {mo_best_solution['auroc']:.4f}")
print(f"  Brier: {mo_best_solution['brier']:.4f}")
print(f"  Cross-dataset deg: {mo_degradation:.4f}")

print(f"\n✓ Multi-objective results loaded for comparison")

In [ ]:
# ========================================
# COMPARISON TABLE
# ========================================

print("="*80)
print("SINGLE-OBJECTIVE vs MULTI-OBJECTIVE COMPARISON")
print("="*80)

# Create comparison table
comparison = pd.DataFrame({
    'Metric': ['PR-AUC', 'AUROC', 'Brier', 'Cross-Dataset Degradation'],
    'Single-Obj (40 evals)': [
        best_solution['pr_auc'],
        best_solution['auroc'],
        best_solution['brier'],
        best_solution['cross_dataset_degradation']
    ],
    'Multi-Obj (56 evals)': [
        mo_best_solution['pr_auc'],
        mo_best_solution['auroc'],
        mo_best_solution['brier'],
        mo_degradation
    ]
})

comparison['Difference (SO - MO)'] = (
    comparison['Single-Obj (40 evals)'] - comparison['Multi-Obj (56 evals)']
)

print("\n" + comparison.to_string(index=False))

# Key findings
pr_diff = best_solution['pr_auc'] - mo_best_solution['pr_auc']
print(f"\nKey Findings:")
if pr_diff > 0.001:
    print(f"  ✓ Single-objective achieved {pr_diff:.4f} higher PR-AUC")
elif pr_diff < -0.001:
    print(f"  ✗ Multi-objective achieved {-pr_diff:.4f} higher PR-AUC")
else:
    print(f"  ≈ Comparable PR-AUC performance (diff: {pr_diff:.4f})")

# Save comparison
comparison.to_csv(output_dir / "comparison_so_vs_mo.csv", index=False)
print(f"\n✓ Comparison saved: {output_dir / 'comparison_so_vs_mo.csv'}")

In [ ]:
# ========================================
# CONVERGENCE PLOT
# ========================================

plt.figure(figsize=(10, 6))
plt.plot(results_df['evaluation_id'], results_df['best_pr_auc_so_far'],
         marker='o', linewidth=2, markersize=4, label='Single-Objective')
plt.axhline(y=best_solution['pr_auc'], color='r', linestyle='--',
            label=f"Best: {best_solution['pr_auc']:.4f}")
plt.axhline(y=mo_best_solution['pr_auc'], color='b', linestyle=':',
            label=f"Multi-Obj Best: {mo_best_solution['pr_auc']:.4f}")
plt.xlabel("Evaluation Number", fontsize=12)
plt.ylabel("Best PR-AUC So Far", fontsize=12)
plt.title("Single-Objective BO Convergence", fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(output_dir / "convergence_pr_auc.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Convergence plot saved: {output_dir / 'convergence_pr_auc.png'}")

In [ ]:
# ========================================
# SCATTER PLOTS: PR-AUC vs OTHER METRICS
# ========================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics_compare = ['auroc', 'brier', 'cross_dataset_degradation']
titles = ['AUROC', 'Brier Score (lower better)', 'Cross-Dataset Degradation (lower better)']

# Detect degradation column in pareto_front
if 'cross_dataset_degradation' in pareto_front.columns:
    pareto_degradation_col = 'cross_dataset_degradation'
elif 'robustness' in pareto_front.columns:
    pareto_degradation_col = 'robustness'
else:
    pareto_degradation_col = 'robustness_degradation'

for ax, metric, title in zip(axes, metrics_compare, titles):
    # Multi-objective Pareto front
    if metric == 'cross_dataset_degradation':
        ax.scatter(pareto_front['pr_auc'], pareto_front[pareto_degradation_col],
                   c='blue', alpha=0.6, s=80, label='Multi-Obj Pareto', edgecolors='black')
    else:
        ax.scatter(pareto_front['pr_auc'], pareto_front[metric],
                   c='blue', alpha=0.6, s=80, label='Multi-Obj Pareto', edgecolors='black')

    # Single-objective best
    ax.scatter(best_solution['pr_auc'], best_solution[metric],
               c='red', marker='*', s=400, label='Single-Obj Best',
               edgecolors='black', linewidths=2)

    ax.set_xlabel('PR-AUC', fontsize=12)
    ax.set_ylabel(title, fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / "comparison_scatter.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Scatter plots saved: {output_dir / 'comparison_scatter.png'}")

In [ ]:
# ========================================
# BUDGET EFFICIENCY ANALYSIS
# ========================================

print("="*80)
print("BUDGET EFFICIENCY ANALYSIS")
print("="*80)

print(f"\nSingle-Objective:")
print(f"  Budget: {TOTAL_BUDGET} evaluations")
print(f"  Best PR-AUC: {best_solution['pr_auc']:.4f}")
print(f"  Found at evaluation: {best_solution['evaluation_id']:.0f}")

mo_total = len(mo_results)
print(f"\nMulti-Objective:")
print(f"  Budget: {mo_total} evaluations")
print(f"  Best PR-AUC: {mo_best_solution['pr_auc']:.4f}")

print(f"\nBudget Savings:")
savings = mo_total - TOTAL_BUDGET
print(f"  Single-obj used {savings} fewer evaluations ({savings/mo_total*100:.1f}% reduction)")

# Estimated time savings (assuming 90 min per eval)
time_saved_hours = savings * 1.5
print(f"  Estimated time saved: ~{time_saved_hours:.1f} hours")
print(f"  Plus 84 hours saved by reusing multi-objective results")
print(f"  Total time savings: {time_saved_hours + 84:.1f} hours")

In [ ]:
# ========================================
# EXPORT BEST SOLUTION
# ========================================

# Export best solution for zero-shot evaluation
best_hyperparams = {
    'learning_rate': best_solution['learning_rate'],
    'weight_decay': best_solution['weight_decay'],
    'dropout': best_solution['dropout'],
    'augmentation_strength': best_solution['augmentation_strength'],
    'unfreeze_fraction': best_solution['unfreeze_fraction']
}

import pickle
with open(output_dir / "best_hyperparams.pkl", 'wb') as f:
    pickle.dump(best_hyperparams, f)

print("Best hyperparameters exported for zero-shot evaluation:")
print(f"  {output_dir / 'best_hyperparams.pkl'}")

# Also save as JSON for easy viewing
with open(output_dir / "best_hyperparams.json", 'w') as f:
    json.dump(best_hyperparams, f, indent=2)

print(f"  {output_dir / 'best_hyperparams.json'}")

In [ ]:
# ========================================
# FINAL REPORT
# ========================================

report = f"""
# Single-Objective Bayesian Optimization Results

## Experiment Configuration
- **Objective**: Maximize PR-AUC
- **Budget**: {TOTAL_BUDGET} evaluations ({N_INITIAL} initial + {N_ITERATIONS}×{BATCH_SIZE} BO)
- **Method**: BoTorch qExpectedImprovement
- **Date**: {datetime.now().strftime('%Y-%m-%d %H:%M')}

## Best Solution Found
- **PR-AUC**: {best_solution['pr_auc']:.4f}
- **AUROC**: {best_solution['auroc']:.4f}
- **Brier**: {best_solution['brier']:.4f}
- **Cross-Dataset Degradation**: {best_solution['cross_dataset_degradation']:.4f}

### Hyperparameters
- Learning rate: {best_solution['learning_rate']:.9f}
- Weight decay: {best_solution['weight_decay']:.9f}
- Dropout: {best_solution['dropout']:.3f}
- Augmentation strength: {best_solution['augmentation_strength']:.3f}
- Unfreeze fraction: {best_solution['unfreeze_fraction']:.3f}

## Comparison with Multi-Objective
- **Single-Obj PR-AUC**: {best_solution['pr_auc']:.4f}
- **Multi-Obj PR-AUC**: {mo_best_solution['pr_auc']:.4f}
- **Difference**: {best_solution['pr_auc'] - mo_best_solution['pr_auc']:.4f}
- **Budget Savings**: {savings} evaluations (~{time_saved_hours:.1f} hours)
- **Plus**: 84 hours saved by reusing multi-objective results

## Files Generated
- `evaluations.csv`: All {len(results_df)} evaluations
- `best_hyperparams.pkl`: Best solution for retraining
- `best_hyperparams.json`: Human-readable best solution
- `convergence_pr_auc.png`: Convergence plot
- `comparison_scatter.png`: Visual comparison with multi-objective
- `comparison_so_vs_mo.csv`: Detailed comparison table
"""

# Save report
with open(output_dir / "REPORT.md", 'w') as f:
    f.write(report)

print("Final report saved:")
print(f"  {output_dir / 'REPORT.md'}")
print("\n" + "="*80)
print("ALL DONE! ✓")
print("="*80)